In [13]:
import numpy as np
import matplotlib.pyplot as plt
from keysight import pwdatatools as pwdt
from scipy.special import ellipk



In [12]:
print(pwdt.__version__)

0.11.0


In [6]:
#Import Data
Frequency_R2, MagS21_R2 = np.loadtxt('Run_2.csv', delimiter = '\t',skiprows=1, unpack=True)

c=3*10**8

In [15]:
ds = pwdt.read_file_as_group(r'C:\Users\Benji\GroundPlaneSingleResonator_wrk\data\Run12.ds')

ModuleNotFoundError: Cannot read ADS dataset. Either set an HPEESOF_DIR environment variable or install the keysight-ads-dataset library

In [9]:
print(pwdt.dirs)

AttributeError: module 'keysight.pwdatatools' has no attribute 'dirs'

In [ ]:
idx = MagS21_R2.argmin()
print(MagS21_R2[idx])
res_f = Frequency_R2[idx]

### Theortical Frequency Prediction

The resonant frequency of a CPW resonator can be found using the followng equation[1]:
$$
f_0 = \frac{c}{\sqrt{\epsilon_{eff}}}\frac{1}{2l}
$$
where $c$ is the speed of light, $\epsilon_{eff}$ is th effective permitivity of CPW line and $l$ is length of the resonantor with a factor of $2$ as it is a half wavelength resonator which implies $2l = \lambda_0$, the fundamental mode. 

To calculate the the effective permeability the following equation is used[2].
$$
\epsilon_{eff} = \frac{\epsilon_r + 1}{2} + \frac{\epsilon_r - 1}{2} \frac{1}{\sqrt{1+12d/W}}
$$  
For a microstrip line built on top a single substrate.

In [ ]:
def find_effective_permeability(e_r,d,W):
    e_e = ((e_r+1)/(2))+((e_r-1)/(2))*(1/(np.sqrt(1+(12*d/W))))
    return e_e

def find_resonator_frequency(e_r,d,W,l):
    e_e = find_effective_permeability(e_r,d,W)

    f0 = (c/(np.sqrt(e_e)))*(1/(2*l))
    return f0

theo_res_f = find_resonator_frequency(11.6,270*(10**-6),10*(10**-6),12*(10**-3))
theo_res_f = theo_res_f/(10**9)
print(f'{theo_res_f} GHz')

In [ ]:
plt.figure().set_figwidth(15)
plt.title('Resonant Frequency Sweep')
plt.xlabel(r'Frequency $(\mathrm{GHz})$')
plt.ylabel(r'$| S_{21} |$ ($\mathrm{dB}$)')
plt.plot(Frequency_R2,MagS21_R2,label = 'Simulation Data')
plt.axvline(x=4.6385,c='r',ls='--',label='Expected $f_0$')
plt.axvline(x=4.8678,c='g',ls='--',label='Theortical $f_0$')
plt.text(6, -0.05, rf'Simulation $f_0$ =  {res_f:.4f} GHz'
         f'\n Expected $f_0$ = 4.6388 GHz'
         f'\n Theortical $f_0$ = {theo_res_f:.4f} GHz ',bbox=dict(facecolor='white', alpha=1))
plt.grid()
plt.legend(loc='best')
plt.show()

## Discussion


In [ ]:
percent_error = -((res_f-4.6388)/res_f)*100
print(percent_error)

### Conformal Mapping to measure Inductance and Capacitance 

Conformal mapping is a method of solving or the capcitance $C_l$ and Inductance $L_l$ per unit length. These can be related to measure the phase velocity $v_p$. As the phase velocity $v_p = \frac{c}{\sqrt{\epsilon_{eff}}}$ the result can be used to find $f_0$.

The Inductance are found using the following equations[3]:
$$
L_l = \frac{\mu_0}{4}\frac{K(k'_0)}{K(k_0)}
$$
$$
C_l = 4\epsilon_0 \epsilon_{eff} \frac{K(k_0)}{k'_0}
$$

Here $K$ denotes the elliptic integral of the first kind with the arguments

$$
k_0 = \frac{w}{w+2s}
$$
$$
k'_0 = \sqrt{1-k^2_0}
$$